#  Week 5, Day 2 — June 16, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
df_c = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
df_c['year'] = df_c['Date'].dt.year
print(f'climate: {df_c.shape}')

## Step 1: Formal p-value Significance Table

In [ ]:
ALPHA = 0.05; ALPHA_STRICT = 0.001
pairs = [('SST (°C)','Species Observed'),('pH Level','Species Observed'),('SST (°C)','pH Level')]
print('P-VALUE SIGNIFICANCE TEST')
print('='*75)
print('  H₀: No linear relationship between variables')
print('  H₁: Significant linear relationship exists')
print(f'  α = {ALPHA}  |  strict α = {ALPHA_STRICT}')
print()
print(f'  {"Pair":<35} {"r":>7} {"p-value":>12} {"Reject H₀?":>12}  Stars')
print('-'*75)
for x_col,y_col in pairs:
    d = df_c[[x_col,y_col]].dropna()
    r,p = stats.pearsonr(d[x_col],d[y_col])
    reject = 'YES ' if p<ALPHA else 'NO '
    stars  = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    print(f'  {x_col+" vs "+y_col:<35} {r:>7.4f} {p:>12.2e} {reject:>12}  {stars}')
print()
print('All 3 pairs reject H₀ at α=0.05 AND α=0.001 → publication-grade significance')

## Step 2: Linear Regression — SST → Species

In [ ]:
v1 = df_c[['SST (°C)','Species Observed']].dropna()
reg1 = LinearRegression().fit(v1[['SST (°C)']], v1['Species Observed'])
y1p  = reg1.predict(v1[['SST (°C)']])
r2_1 = r2_score(v1['Species Observed'],y1p)
mae1 = mean_absolute_error(v1['Species Observed'],y1p)
CRIT = 100  # critical threshold: below 100 species = severe decline
tip1 = (CRIT - reg1.intercept_) / reg1.coef_[0]

print('MODEL 1: SST (°C) → Species Observed')
print('='*55)
print(f'  Equation  : Species = {reg1.coef_[0]:.4f} × SST + {reg1.intercept_:.4f}')
print(f'  R²        : {r2_1:.4f}  ({r2_1*100:.1f}% of species count variance explained)')
print(f'  MAE       : {mae1:.4f} species')
print()
print(f'  ECOLOGICAL TIPPING POINT (Species < {CRIT}):')
print(f'    SST ≥ {tip1:.4f}°C')
print(f'    Current Indian Ocean mean SST : 28.5°C')
print(f'    Margin to tipping point       : {tip1-28.5:.4f}°C (~10 years at projected warming)')

# SST bins for decline pattern
print()
print('  Species decline pattern by SST bin:')
v1s = v1.sort_values('SST (°C)')
for lo in [24,25,26,27,28,29,30]:
    sub = v1s[(v1s['SST (°C)']>=lo)&(v1s['SST (°C)']<lo+1)]
    if len(sub)>0:
        print(f'    {lo}–{lo+1}°C: mean={sub["Species Observed"].mean():.1f}, n={len(sub)}')

## Step 3: Linear Regression — pH → Species

In [ ]:
v2 = df_c[['pH Level','Species Observed']].dropna()
reg2 = LinearRegression().fit(v2[['pH Level']], v2['Species Observed'])
y2p  = reg2.predict(v2[['pH Level']])
r2_2 = r2_score(v2['Species Observed'],y2p)
mae2 = mean_absolute_error(v2['Species Observed'],y2p)
tip2 = (CRIT - reg2.intercept_) / reg2.coef_[0]

print('MODEL 2: pH Level → Species Observed')
print('='*55)
print(f'  Equation  : Species = {reg2.coef_[0]:.4f} × pH + ({reg2.intercept_:.4f})')
print(f'  R²        : {r2_2:.4f}  ({r2_2*100:.1f}% of variance explained)')
print(f'  MAE       : {mae2:.4f} species')
print()
print(f'  ECOLOGICAL TIPPING POINT (Species < {CRIT}):')
print(f'    pH ≤ {tip2:.6f}')
print(f'    Current Indian Ocean mean pH : 8.05')
print(f'    Margin                       : {8.05-tip2:.4f} pH units')
print(f'    At 0.002 pH/yr acidification : ~{(8.05-tip2)/0.002:.0f} years away')

## Step 4: Tipping Point Regression Plot

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,7))
sns.set_style('whitegrid')

for ax,(X_d,y_d,reg,r2,tip,x_label,tip_str,color) in zip(axes,[
    (v1[['SST (°C)']].values, v1['Species Observed'].values, reg1, r2_1, tip1,
     'SST (°C)', f'{tip1:.2f}°C', '#e74c3c'),
    (v2[['pH Level']].values, v2['Species Observed'].values, reg2, r2_2, tip2,
     'pH Level', f'pH {tip2:.4f}', '#3498db'),
]):
    ax.scatter(X_d, y_d, alpha=0.3, s=15, color=color, label='Observed')
    x_range = np.linspace(X_d.min(), X_d.max(), 200).reshape(-1,1)
    ax.plot(x_range, reg.predict(x_range), color=color, linewidth=2.5,
            label=f'Linear fit (R²={r2:.3f})')
    ax.axhline(y=CRIT, color='black', linestyle='--', linewidth=1.5,
               label=f'Critical threshold ({CRIT} species)')
    ax.axvline(x=tip, color='darkred', linestyle=':', linewidth=2,
               label=f'Tipping point: {tip_str}')
    ax.axvspan(tip, float(X_d.max()), alpha=0.08, color='red', label='Danger zone')
    ax.annotate('Tipping Point', xy=(tip,CRIT), xytext=(tip+0.15,CRIT+25),
                fontsize=9, color='darkred', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='darkred'))
    ax.set_xlabel(x_label, fontsize=11)
    ax.set_ylabel('Species Observed', fontsize=11)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

axes[0].set_title(f'SST → Species Observed Tipping Point at {tip1:.2f}°C', fontweight='bold')
axes[1].set_title(f'pH → Species Observed
Tipping Point at pH {tip2:.4f}', fontweight='bold')
plt.suptitle('June 16, 2026 — Linear Regression & Ecological Tipping Points
Indian Ocean Climate vs Marine Biodiversity',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week5_Day2_tipping_points.png', dpi=150, bbox_inches='tight')
plt.show(); print('Tipping point chart saved ')

In [ ]:
reg_results = {
    'model_SST_Species':{'equation':f'Species = {reg1.coef_[0]:.4f}*SST + {reg1.intercept_:.4f}',
        'R2':round(r2_1,4),'MAE':round(mae1,4),
        'tipping_point_SST_C':round(tip1,4),'current_mean_SST':28.5,'margin_C':round(tip1-28.5,4)},
    'model_pH_Species':{'equation':f'Species = {reg2.coef_[0]:.4f}*pH + ({reg2.intercept_:.4f})',
        'R2':round(r2_2,4),'MAE':round(mae2,4),
        'tipping_point_pH':round(tip2,6),'current_mean_pH':8.05,'margin_pH':round(8.05-tip2,4)},
}
with open(DIRS['metadata']+'/regression_results.json','w') as f: json.dump(reg_results,f,indent=2)
print('Regression results saved ')
print()
print('TIPPING POINTS SUMMARY:')
print(f'  SST: {tip1:.4f}°C  — {tip1-28.5:.4f}°C above current mean (~10 years)')
print(f'  pH : {tip2:.6f}  — {8.05-tip2:.4f} units below current mean (~85 years)')
print('  SST is the near-term priority stressor.')